# Metronome Compass Calibration

A Python port of RNGReporter's HGSS **Seed to Time** verification panel, built for
gathering Metronome-compass calibration data.

Give it a target datetime + delay and a search window, and it enumerates every candidate
seed nearby.  For each seed it reports:

- **Roamer relocation** — where Raikou / Entei / Latios(Latias) move to when the save is
  reloaded (given where they are now).
- **Elm phone-call sequence** — the `P`/`E`/`K` calls you can read off in-game.

Two sections identify the seed two ways: **Section A** from roamer routes + Elm calls
(`a_seed`), **Section B** from the Metronome battle (`b_seed`).  All logic lives in
`utils/calibration_tools.py`.

## Keyboard fixup

The `2` and `w` keys on my keyboard are flaky, so every `input()` prompt below accepts
`\T` for `2` and `\V` for `w` (substituted before the value is used).  It's applied
automatically at each interactive step — ipykernel resets `input` once per cell, so the
library re-installs the fixup at every prompt.  To add pairs, edit `INPUT_SUBS` in
`utils/calibration_tools.py`.


In [25]:
%load_ext autoreload
%autoreload 2
import datetime as dt
from utils.calibration_tools import (
    # Section A -- roamer routes + Elm calls
    generate_roamer_candidates_near,
    print_roamer_candidates,
    identify_seed,
    # Section B -- Metronome-compass battle
    generate_candidates_near,
    print_candidates,
    narrow_candidates,
    prompt_magikarp,
    # Persist a run
    save_compass_run,
    # Timer calibration math
    calibrate_timer,
)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Section A — Roamer + Elm identification  (→ `a_seed`)

**Configure** your target datetime/delay, the search window, and each roamer's **current**
route (where it is *now*, before the reset) plus whether it's still roaming.  Use `0` for a
current route you don't know or care about.  Variables are `a_`-prefixed so they won't clash
with Section B.

**Identify** (after loading the save, read the roamer map and Elm phone):

1. **Roamer routes** — one number per *roaming* legendary in **R E L** order (e.g. `38 42 11`);
   `.` leaves a roamer unconstrained.  Only roamers marked `present` are expected.
2. If more than one candidate matches, **Elm calls** — type `P`/`E`/`K` as you hear each
   call (matched as a substring, since RNG may advance first); other characters are ignored.
   Type `M` to pick a candidate by number instead.
3. The single surviving row is saved to **`a_seed`** (integer seed is `a_seed["seed"]`).


In [29]:
# --- Section A: roamer / Elm target + current roamer state ---
a_target_time    = dt.datetime(2025, 7, 24, 14, 45, 55)
a_target_delay   = 681
a_seconds_window = 1        # +/- X seconds
a_delay_window   = 60       # +/- Y delays
a_match_parity   = True     # only delays with target_delay's even/odd parity
a_display_limit  = 40       # rows to print (None = all)

# Each roamer's CURRENT route (before the reset) and whether it is still roaming.
a_prev_routes = {"r": 36, "e": 38, "l": 28}
a_present     = {"r": True, "e": True, "l": True}

a_candidates = generate_roamer_candidates_near(
    a_target_time, a_target_delay, a_seconds_window, a_delay_window,
    prev_routes=a_prev_routes, present=a_present, match_parity=a_match_parity,
)

# Interactively pin down the seed: roamer routes -> Elm calls -> (M) manual pick.
a_seed = identify_seed(a_candidates, a_present, display_limit=a_display_limit)
# GOALS: EKP, KPEK, PEKK, EKKP

Observed roamer routes (R E L, space-separated, . = any):  30 46 11



Observed R=30 E=46 L=11  ->  3 / 183 candidate(s) match

3 candidate seed(s)

        Seed                 Time   Delay    dD   ds    R   E   L   #  Elm
  0x0B0E02BE  2025-07-24 14:45:54     677    -4   -1   30  46  11   3  PEPKKPEPEPEPEPE
  0x0C0E02BE  2025-07-24 14:45:55     677    -4   +0   30  46  11   3  KKKEKEKEKEKKPEK
  0x0D0E02BE  2025-07-24 14:45:56     677    -4   +1   30  46  11   3  EPEPKKEEPKPKKPK


Elm calls (type P/E/K as heard; M = pick manually):  kek


Elm calls so far: KEK
1 candidate seed(s)

        Seed                 Time   Delay    dD   ds    R   E   L   #  Elm
  0x0C0E02BE  2025-07-24 14:45:55     677    -4   +0   30  46  11   3  KKKEKEKEKEKKPEK

=== Seed identified ===
1 candidate seed(s)

        Seed                 Time   Delay    dD   ds    R   E   L   #  Elm
  0x0C0E02BE  2025-07-24 14:45:55     677    -4   +0   30  46  11   3  KKKEKEKEKEKKPEK


## Section B — Expedition-style Seed Identification  (→ `b_seed`)

Ported from the Metronome Compass Testing notebook.  Generate every candidate seed near a
target `(time, delay)`, each with its precomputed Metronome battle path, then walk the real
battle turn by turn — candidates whose path diverges from what you observe drop out until a
single seed remains, saved as **`b_seed`** (the whole row).

Config uses `b_`-prefixed names so it won't clash with Section A.  The Metronome user's
movepool besides Metronome is `DEFAULT_EXTRA_MOVES` in `utils/calibration_tools.py`; edit it
there if it changes.  (Seeds here use the same year-correct `seed_for` as Section A.)

Magikarp's **level and gender are asked at run time** (they change each battle); the Metronome user's own gender is the stable `b_metronome_user_is_female` config.


In [30]:
# --- Section B: Metronome-compass target ---
# b_target_time     = dt.datetime(2025, 7, 24, 14, 49, 0)
b_target_time = a_target_time + dt.timedelta(seconds=182)
b_target_delay    = 11000
b_seconds_window  = 2         # +/- X seconds
b_delay_window    = 1000       # +/- Y delays
b_metronome_only  = False     # True = Metronome-only user; False = + DEFAULT_EXTRA_MOVES
b_metronome_user_is_female = True   # the Metronome user's gender (rarely changes)

# Magikarp's level + gender change every run, so prompt for them at execution time.
# opposite_gender is derived relative to the Metronome user's gender above.
b_magikarp_level, b_opposite_gender = prompt_magikarp(b_metronome_user_is_female)

b_candidates = generate_candidates_near(
    b_target_time, b_target_delay, b_seconds_window, b_delay_window,
    magikarp_level=b_magikarp_level, opposite_gender=b_opposite_gender,
    metronome_only=b_metronome_only,
)

# Walk the real battle turn by turn; candidates diverging from what you observe drop out.
b_seed = narrow_candidates(b_candidates, b_magikarp_level, b_opposite_gender,
                           metronome_only=b_metronome_only)


Magikarp level:  15
Magikarp gender (M/F):  M


10005 candidate seed(s)

        Seed                 Time   Delay    dD   ds  Path
  0x110E2B11  2025-07-24 14:48:57   11000    +0   +0  KtkhM184h KspM038! KspM450h KtkhM247h KspM013 Ktk!h KtkhM396h KspM162h KtkhM463hBD Ktk-M226_
  0x100E2B11  2025-07-24 14:48:56   11000    +0   -1  KtkhM198h! KspM237h KspM107 KtkhM283h?_
  0x120E2B11  2025-07-24 14:48:58   11000    +0   +1  KtkhM170 KspM116 KtkhM176CVRock KtkhM458h KtkhM455 KtkhM220 Ktk-M060!~ Ktk-M375 KspM107 SCFZKspM200h
  0x0F0E2B11  2025-07-24 14:48:55   11000    +0   -2  KtkhM212 KspM207h CFZM376h SCFZKspM391 KspM395h KtkhM218h KspM122h~ KspM130 PARh KtkhM006h
  0x130E2B11  2025-07-24 14:48:59   11000    +0   +2  KtkhM156 Ksp Ksp KspM122h~ KspM404h KspM072h KtkhM045h KtkhM048h KspM190h~ KspM142h
  0x110E2B10  2025-07-24 14:48:57   10999    -1   +0  KspM091 Ksph Ktk!M051h KspM346 KspM139h KspM211h KtkhM256 KspM255h KspM072h KtkhM115
  0x110E2B12  2025-07-24 14:48:57   11001    +1   +0  KtkhM302h KtkhM138 Ktk-M223h~ KspM049h KspM4

  Magikarp used? (sp/tk):  tk
  Tackle hit, crit, or miss? (h/!/-):  h
  Metronome selected? (move name or M###):  Double Slap
  Hit, crit, or miss? (h/!/-):  h
  Next hit? (h/!/d for done):  h
  Next hit? (h/!/d for done):  h
  Next hit? (h/!/d for done):  h
  Next hit? (h/!/d for done):  d



1 / 10005 seeds remain -- next is turn 2
        Seed   Delay    dD  predicted turn 2
  0x110E2B1E   11013   +13  KtkhM256           (Swallow)

Seed identified: 0x110E2B1E  time=2025-07-24 14:48:57  delay=11013  dD=+13
Full path: KtkhM003hhhh KtkhM256 KspM222Mag10h KspM311h KspM389 Ktk-M120h_
Remaining Metronome moves (turn 2+):
  Turn 2: Swallow (M256)
  Turn 3: Magnitude (M222)
  Turn 4: Weather Ball (M311)
  Turn 5: Sucker Punch (M389)
  Turn 6: Selfdestruct (M120)


## Section C — Save the run  (→ `data/compass_runs.jsonl`)

Records this calibration run — both identified seeds (`a_seed`, `b_seed`) plus the metadata
below — as one JSON line appended to `data/compass_runs.jsonl`.

You're prompted for a **run tag** (e.g. `300s Samwise`, `11000d Work`), the **target timer
delay**, the **target timer calibration**, and free-form **notes**.  Leaving the tag / delay
/ calibration blank re-uses the previous run's value (notes never default).  The record is
pretty-printed and confirmed (`y`/`n`) before it's written.


In [31]:
# --- Section C: append this run to data/compass_runs.jsonl ---
run_record = save_compass_run(a_seed, b_seed)


Run tag [11000d Smeagol]:  
Target timer delay [185000]:  181913
Target timer calibration [-5000]:  
Notes:  



{
  "saved_at": "2026-09-05T15:35:57",
  "tag": "11000d Smeagol",
  "target_timer_delay": 181913,
  "target_timer_calibration": -5000,
  "notes": "",
  "a_seed": {
    "seed": 202244798,
    "seed_hex": "0x0C0E02BE",
    "time": "2025-07-24T14:45:55",
    "delay": 677,
    "sec_delta": 0,
    "delay_delta": -4,
    "r_route": 30,
    "e_route": 46,
    "l_route": 11,
    "rng_calls": 3,
    "elm": "KKKEKEKEKEKKPEK"
  },
  "b_seed": {
    "seed": 286141214,
    "seed_hex": "0x110E2B1E",
    "time": "2025-07-24T14:48:57",
    "delay": 11013,
    "sec_delta": 0,
    "delay_delta": 13,
    "path_str": "KtkhM003hhhh KtkhM256 KspM222Mag10h KspM311h KspM389 Ktk-M120h_"
  }
}



Save this run? (y/n):  y


Saved to data/compass_runs.jsonl


## Section D — Timer calibration math  (delay/calibration ↔ seed_b frame)

Fits the collected runs to answer: **given a timer countdown, what `F_b` frame will I hit?**

**Model.**  With `M = target_timer_delay + target_timer_calibration` (ms, calibration signed),

$$F_b = \beta\,M + \alpha$$

- **β** — frames per ms (`= rate/1000`, rate ≈ 59.8261 Hz).  The *slope*.
- **α** — the *intercept*.  It absorbs **every** fixed offset at once: the ~5 s between
  timer expiry and battle-seed generation **and** the "loading advances the clock but not
  the frame counter" y-intercept.  Those two are degenerate — only their sum is
  measurable — so `α` (equivalently, your calibration) simply *is* that sum.  You never
  have to split them.

**Two slope estimators, combined:**

- *within-run* `rate = (F_b − F_a)/(T_b − T_a)` — works from a single run off a ~180 s /
  ~10 000-frame lever arm.  Only this one is exposed to the **±1 s** timestamp granularity,
  but that error *averages down* across runs (√N) and is therefore reducible.
- *between-run* Theil–Sen regression of `F_b` on `M` — immune to the ±1 s issue, but needs
  your `M` values to vary.  Used as a cross-check.

**Two kinds of uncertainty, reported separately:**

- **calibration/rate uncertainty** (reducible) — shrinks as you gather runs; pivots about
  the delays you've actually measured, so it only grows as you *extrapolate*.
- **physical jitter** (irreducible) — the frame-rate wobble that worsens with longer
  countdowns (modelled `σ ≈ c·√M`).  This is the real spread you'll face even with perfect
  calibration, and it sets your hit probability.  It's the ± in the min/expected/max below.

Collect runs across a **spread of countdowns** (some short, some long) — that pins β/α and
lets the jitter-vs-`M` growth be measured.


In [32]:
# --- Section D: fit the collected runs and predict / solve ---
model = calibrate_timer()   # reads data/compass_runs.jsonl, prints the report

# What frame will a given commanded countdown land on? (min / expected / max)
d_delay       = 185000      # target_timer_delay to check (ms)
d_calibration = -5000           # target_timer_calibration (ms, signed)
p = model["predict"](d_delay, d_calibration)   # range = expected +/- (rate_band + 2*jitter)
print(f"\nPredict delay={d_delay} cal={d_calibration}  (M={p['M']}):")
print(f"  expected F_b = {p['expected']:.1f}   ~95% range [{p['lo']:.1f}, {p['hi']:.1f}]")
print(f"    physical jitter  +/- {p['jitter']:.1f} frames (irreducible, 1 sigma)")
print(f"    rate/calib band  +/- {p['rate_band']:.1f} frames (reducible; grows as you extrapolate)")

# Probability of actually landing on a target frame (within +/- tolerance frames).
d_target_fb  = 11000
d_tolerance  = 0.5    # 0.5 = exactly that frame; raise if a window of frames is acceptable
hp = model["hit_probability"](d_delay, d_calibration, d_target_fb, tolerance=d_tolerance)
best = model["hit_probability"](d_delay, d_calibration, d_target_fb,
                                 tolerance=d_tolerance, perfect_calibration=True)
print(f"\nP(hit F_b={d_target_fb} +/-{d_tolerance}) with delay={d_delay} cal={d_calibration}:")
print(f"  expected off by {hp['delta']:+.1f} frames; sigma_total = {hp['sigma_total']:.1f} "
      f"(jitter {hp['sigma_jitter']:.1f} (+) calib {hp['sigma_calib']:.1f})")
print(f"  probability = {hp['p']*100:.2f}%   (perfect calibration: {best['p']*100:.2f}%)")

# Inverse: what countdown lands on a target frame? (hold calibration, adjust delay)
d_target_fb = 11000
sol = model["solve"](d_target_fb, calibration=d_calibration)
print(f"\nTo hit F_b={d_target_fb} with calibration={d_calibration}: "
      f"delay = {sol['delay']:.0f} ms  (M = {sol['M']:.0f})")


=== Timer calibration  (3 run(s), 3 timed) ===

  F_b = beta*M + alpha,   M = delay + calibration (ms)
  slope  beta  = 0.056817 frames/ms (= 56.8169 Hz)   [within-run rate]
         within-run rate 56.8169 +/- 0.0721 Hz (1σ, from 3 run(s))
         F_b-vs-M slope  56.3868 Hz  (Theil-Sen cross-check)
  intercept alpha = +952.7 frames (+16768 ms)   (the ~5 s battle delay + load y-intercept, combined)
  jitter  ~ 0.023*sqrt(M) frames (RMS residual 9.8)

  tag                     M      Fa      Fb      dF    dt     rate
  11000d Smeagol     185000     673   11469   10796   190   56.821
  11000d Smeagol     180000     651   11166   10515   185   56.838
  11000d Smeagol     176913     677   11013   10336   182   56.791

Predict delay=185000 cal=-5000  (M=180000):
  expected F_b = 11179.8   range [11160.0, 11199.5]
    physical jitter  +/- 9.8 frames
    rate/calib band  +/- 0.0 frames (grows as you extrapolate)

To hit F_b=11000 with calibration=-5000: delay = 181836 ms  (M = 176836)
